In [1]:
# Notebook: 04_portfolio_sample_size
# Realistic-cell refinement of the Metric I power study. The adjustment directions
# are far from balanced (up 10.7, hold 80.7, down 8.6 per cent), so within a band the
# hold cell is about nine times larger than the up and down cells, and a downgrade-arm
# test is governed by the smaller down cell. This notebook therefore reports the
# operating characteristics against the TOTAL portfolio size N, allocating cells by the
# disclosed direction mix under an even spread across bands, which is the decision
# relevant quantity for a practitioner sizing a test.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

RNG = np.random.default_rng(20260419)
BASE = os.path.join("..")
FIG = os.path.join(BASE, "results", "figures")
TAB = os.path.join(BASE, "results", "tables")

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.30",
                     "axes.linewidth": 0.8, "grid.color": "0.88",
                     "grid.linewidth": 0.6, "savefig.bbox": "tight",
                     "figure.dpi": 150})


def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, name + "." + ext), dpi=600)
    plt.close(fig)


rates = pd.read_csv(os.path.join(BASE, "data", "disclosed_rates.csv"))
rates["p"] = rates["default_rate_pct"] / 100.0
P = {(r.band, r.direction): (np.nan if pd.isna(r.p) else float(r.p))
     for r in rates.itertuples()}
mix = pd.read_csv(os.path.join(BASE, "data", "direction_mix.csv"))
PI = {r.direction: r.proportion_pct / 100.0 for r in mix.itertuples()}
BANDS = ["BB+", "BB", "BB-", "B+", "B", "B-"]
G = len(BANDS)
ALPHA = 0.05


def two_prop_p(k1, n1, k0, n0, side):
    p1, p0 = k1 / n1, k0 / n0
    pp = (k1 + k0) / (n1 + n0)
    se = np.sqrt(pp * (1 - pp) * (1.0 / n1 + 1.0 / n0))
    se = np.where(se == 0, np.nan, se)
    z = (p1 - p0) / se
    return np.where(np.isnan(se), 1.0, stats.norm.sf(side * z))


# cells allocated by direction mix under an even band spread: n_{g,d} = N * (1/G) * pi_d
def cell_n(Ntot, d):
    return Ntot * (1.0 / G) * PI[d]


tracked = [("down " + g, g, "D", "H", +1) for g in BANDS] + \
          [("up B+", "B+", "U", "H", -1)]

N_GRID = [10000, 20000, 40000, 70000, 100000, 150000, 250000, 400000]
R = 4000

power_rows = []
for Ntot in N_GRID:
    draws = {}
    for g in BANDS:
        for d in ("U", "H", "D"):
            p = P[(g, d)]
            n = int(round(cell_n(Ntot, d)))
            draws[(g, d)] = (None if (np.isnan(p) or n < 1)
                             else (RNG.binomial(n, p, size=R), n))
    for (lab, g, d1, d0, side) in tracked:
        c1, c0 = draws[(g, d1)], draws[(g, d0)]
        if c1 is None or c0 is None:
            continue
        (k1, n1), (k0, n0) = c1, c0
        pv = two_prop_p(k1, n1, k0, n0, side)
        power_rows.append({"arm": lab, "N_total": Ntot,
                           "n_small_cell": min(n1, n0),
                           "power": float(np.mean(pv < ALPHA))})
power_df = pd.DataFrame(power_rows)
power_df.to_csv(os.path.join(TAB, "t04_power_by_total_N.csv"), index=False)

# minimum total N for 80% power per arm (interpolated)
minN_rows = []
for (lab, g, d1, d0, side) in tracked:
    sub = power_df[power_df["arm"] == lab].sort_values("N_total")
    xs, ys = sub["N_total"].values, sub["power"].values
    nmin = np.nan
    for i in range(len(xs) - 1):
        if ys[i] < 0.80 <= ys[i + 1]:
            nmin = xs[i] + (0.80 - ys[i]) * (xs[i + 1] - xs[i]) / (ys[i + 1] - ys[i])
            break
    if len(ys) and ys[0] >= 0.80:
        nmin = xs[0]
    sep = (P[(g, d1)] - P[(g, d0)]) * 100
    minN_rows.append({"arm": lab, "true_sep_pp": round(sep, 2),
                      "min_total_N_80pct": (None if np.isnan(nmin) else int(round(nmin)))})
minN_df = pd.DataFrame(minN_rows)
minN_df.to_csv(os.path.join(TAB, "t04_min_total_N.csv"), index=False)

# figure: power vs total portfolio N (grayscale), representative arms
rep = [("down B-", "0.10", "-", "o"),
       ("down B+", "0.10", "--", "s"),
       ("down BB-", "0.45", "-.", "^"),
       ("up B+", "0.45", ":", "D")]
fig, ax = plt.subplots(figsize=(6.2, 4.2))
for lab, c, ls, mk in rep:
    sub = power_df[power_df["arm"] == lab].sort_values("N_total")
    if sub.empty:
        continue
    ax.plot(sub["N_total"], sub["power"], color=c, linestyle=ls, marker=mk,
            markersize=5, markerfacecolor="white", markeredgecolor=c,
            linewidth=1.4, label=lab)
ax.axhline(0.80, color="0.55", linewidth=0.9, linestyle=(0, (1, 1)))
ax.set_xscale("log")
ax.set_xlabel("Total portfolio size N (log scale)")
ax.set_ylabel("Power at one-sided 5% level")
ax.set_ylim(0, 1.02)
ax.legend(frameon=False, loc="lower right", fontsize=9)
save_fig(fig, "fig06_power_by_total_N")

print("Monte Carlo replications per point:", R)
print("\nMinimum TOTAL portfolio N for 80% power (even band spread, cells by direction mix):")
print(minN_df.to_string(index=False))
print("\n(For reference, down-cell size at the stated N is N*(1/6)*0.086.)")


Monte Carlo replications per point: 4000

Minimum TOTAL portfolio N for 80% power (even band spread, cells by direction mix):
     arm  true_sep_pp  min_total_N_80pct
down BB+         0.66            57516.0
 down BB         0.11                NaN
down BB-         0.40           337105.0
 down B+         1.24            57185.0
  down B         1.97            34096.0
 down B-         2.77            28483.0
   up B+         0.25                NaN

(For reference, down-cell size at the stated N is N*(1/6)*0.086.)
